# Perpetuos USDT-M: velas 4h, funding y fechas de listado (validación de Cascada)

Paquete de datos del **protocolo de validación** (`validacion/PROTOCOLO.md`, sección 3).

**Qué baja**
1. Para cada moneda que pasó por el top 50 (`resumen_descarga.csv`): velas de 4 h y funding histórico del **perpetuo USDT-M**.
   - **Binance** primero (`data.binance.vision`, incluye contratos deslistados). Prueba `SYMUSDT`, `1000SYMUSDT` y `1000000SYMUSDT`; los de prefijo 1000 se reescalan al precio de una unidad.
   - **KuCoin** (vía `ccxt`) si Binance no tiene el contrato.
2. Las especificaciones de los contratos de KuCoin (tamaño, mínimo, fecha de apertura).
3. Arma las matrices con índice = apertura de la vela (UTC) y una columna por símbolo, más el funding y la tabla de listados.
4. Calcula la huella SHA-256 de cada archivo y descarga todo en `perpetuos_4h.zip`.

**Qué hacer con el zip:** subilo al proyecto (o adjuntalo en el chat). Tarda ~30–60 min la primera vez; si se corta, volvé a correr desde la celda de descarga y sigue donde quedó.

In [ ]:
!pip -q install ccxt pyarrow

In [ ]:
# ======== CONFIG ========
DESDE     = '2018-06-01'      # precalentamiento de indicadores antes de 2019
HASTA     = '2026-10-01'      # fin del período de prueba (2026-09-30 inclusive)
CARPETA   = 'perpetuos_4h'    # carpeta en tu Drive
GUARDAR_EN_DRIVE = True
MAX_WORKERS = 16
REDESCARGAR = False
RESUMEN_EN_DRIVE = 'universo_top50_4h/resumen_descarga.csv'
# ========================

In [ ]:
import os, io, re, json, zipfile, time, hashlib, requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

if GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = '/content/drive/MyDrive'
else:
    RAIZ = '/content'
OUT = f'{RAIZ}/{CARPETA}'
POR_MONEDA = f'{OUT}/por_moneda'
os.makedirs(POR_MONEDA, exist_ok=True)
T0 = pd.Timestamp(DESDE, tz='UTC'); T1 = pd.Timestamp(HASTA, tz='UTC')
print('Guardando en', OUT)

## 1) Monedas del universo (`resumen_descarga.csv`)

In [ ]:
ruta = f'{RAIZ}/{RESUMEN_EN_DRIVE}'
if not os.path.exists(ruta):
    from google.colab import files
    print('No encontré', ruta, '→ subí resumen_descarga.csv (el del proyecto: top 50 semanal/resumen_descarga 1.csv)')
    subido = files.upload()
    ruta = '/content/' + list(subido)[0]
res = pd.read_csv(ruta).drop_duplicates(['slug', 'symbol']).reset_index(drop=True)
print(len(res), 'monedas')

## 2) Mapeo de tickers (igual que los notebooks anteriores)

In [ ]:
MAPEO = {
    'iota': ['IOTA'], 'polygon-ecosystem-token': ['MATIC', 'POL'], 'polygon': ['MATIC'], 'render': ['RNDR', 'RENDER'],
    'terra-luna': [('LUNA', None, '2022-05-14'), 'LUNC'], 'terra-luna-v2': [('LUNA2', '2022-05-28', None), ('LUNA', '2022-05-28', None)],
    'maker': ['MKR'], 'sky': ['MKR', 'SKY'], 'vaulta': ['EOS', 'A'], 'eos': ['EOS'],
    'bitcoin-cash': ['BCC', 'BCHABC', 'BCH'], 'bitcoin-sv': ['BCHSV', 'BSV'], 'nano': ['NANO', 'XNO'], 'fantom': ['FTM'],
    'sonic': ['S'], 'elrond-egld': ['EGLD'], 'multiversx-egld': ['EGLD'], 'toncoin': ['TON'], 'kucoin-token': ['KCS'],
    'huobi-token': ['HT'], 'htx-dao': ['HTX'], 'ftx-token': ['FTT'], 'celsius': ['CEL'], 'lido-dao': ['LDO'],
    'bitcoin-gold': ['BTG'], 'pepe': ['PEPE'], 'shiba-inu': ['SHIB'], 'bittorrent-new': ['BTTC'], 'bittorrent': ['BTT'],
}
def candidatos(slug, symbol):
    return [c if isinstance(c, tuple) else (c, None, None) for c in MAPEO.get(slug, [symbol])]

def recortar(df, desde, hasta, col='datetime'):
    m = pd.Series(True, index=df.index)
    if desde: m &= df[col] >= pd.Timestamp(desde, tz='UTC')
    if hasta: m &= df[col] < pd.Timestamp(hasta, tz='UTC')
    return df[m]

## 3) Binance USDT-M: lista de contratos publicados, velas y funding

In [ ]:
BASE = 'https://data.binance.vision'
S3 = 'https://s3-ap-northeast-1.amazonaws.com/data.binance.vision'
sess = requests.Session()

def listar(prefijo):
    """Subcarpetas publicadas en data.binance.vision bajo un prefijo (pagina de a 1000)."""
    out, marker = [], ''
    while True:
        r = sess.get(S3, params={'delimiter': '/', 'prefix': prefijo, 'marker': marker}, timeout=60); r.raise_for_status()
        out += re.findall(r'<Prefix>' + re.escape(prefijo) + r'([^/<]+)/</Prefix>', r.text)
        if '<IsTruncated>true</IsTruncated>' not in r.text: break
        marker = prefijo + out[-1] + '/'
    return set(out)

PARES_UM = listar('data/futures/um/monthly/klines/')
print(len(PARES_UM), 'perpetuos/futuros USDT-M publicados en Binance')

def _leer_zip(content, ncols):
    with zipfile.ZipFile(io.BytesIO(content)) as z:
        raw = z.read(z.namelist()[0])
    df = pd.read_csv(io.BytesIO(raw), header=None)
    if not str(df.iloc[0, 0]).strip().lstrip('-').isdigit(): df = df.iloc[1:]
    return df.iloc[:, :ncols]

def _get(url, ncols):
    for i in range(4):
        try:
            r = sess.get(url, timeout=60)
            if r.status_code == 404: return None
            r.raise_for_status()
            return _leer_zip(r.content, ncols)
        except Exception:
            time.sleep(1.5 * (i + 1))
    return None

def _meses():
    hoy = pd.Timestamp.now('UTC').tz_localize(None).normalize()
    return pd.date_range(DESDE, hoy.replace(day=1) - pd.DateOffset(months=1), freq='MS'), hoy

def _bajar_urls(urls, ncols):
    partes = []
    with ThreadPoolExecutor(MAX_WORKERS) as ex:
        for f in as_completed([ex.submit(_get, u, ncols) for u in urls]):
            d = f.result()
            if d is not None and len(d): partes.append(d)
    return pd.concat(partes, ignore_index=True) if partes else None

def binance_velas(par):
    meses, hoy = _meses()
    urls = [f'{BASE}/data/futures/um/monthly/klines/{par}/4h/{par}-4h-{m:%Y-%m}.zip' for m in meses]
    dias = pd.date_range(hoy.replace(day=1), hoy - pd.Timedelta(days=1), freq='D')
    urls += [f'{BASE}/data/futures/um/daily/klines/{par}/4h/{par}-4h-{d:%Y-%m-%d}.zip' for d in dias]
    df = _bajar_urls(urls, 8)
    if df is None: return None
    df.columns = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume']
    df = df.apply(pd.to_numeric, errors='coerce').dropna(subset=['open_time', 'close'])
    df['open_time'] = df['open_time'].astype('int64')
    m = df['open_time'] > 1e14; df.loc[m, 'open_time'] //= 1000
    df['datetime'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
    return df[['datetime', 'open', 'high', 'low', 'close', 'quote_volume']]

def binance_funding(par):
    meses, _ = _meses()
    urls = [f'{BASE}/data/futures/um/monthly/fundingRate/{par}/{par}-fundingRate-{m:%Y-%m}.zip' for m in meses]
    df = _bajar_urls(urls, 3)
    if df is None: return None
    df.columns = ['calc_time', 'intervalo_h', 'tasa']
    df = df.apply(pd.to_numeric, errors='coerce').dropna(subset=['calc_time', 'tasa'])
    df['datetime'] = pd.to_datetime(df['calc_time'].astype('int64'), unit='ms', utc=True).dt.round('min')
    return df[['datetime', 'tasa']]

## 4) KuCoin (si Binance no tiene el contrato) y especificaciones de contratos

In [ ]:
import ccxt
KU = ccxt.kucoinfutures({'enableRateLimit': True}); KU.load_markets()
KU_USDT = {m['base']: s for s, m in KU.markets.items() if m.get('swap') and m.get('linear') and m.get('quote') == 'USDT' and m.get('settle') == 'USDT'}
print(len(KU_USDT), 'perpetuos USDT en KuCoin')

def kucoin_velas(sym):
    since = int(T0.timestamp() * 1000); filas = []; vacios = 0
    while since < KU.milliseconds():
        try:
            lote = KU.fetch_ohlcv(sym, '4h', since=since, limit=200)
        except Exception:
            time.sleep(2); vacios += 1
            if vacios > 5: break
            continue
        if not lote:
            since += 30 * 86400_000; vacios += 1
            if vacios > 120: break
            continue
        filas += lote
        nuevo = lote[-1][0] + 4 * 3600_000
        if nuevo <= since: break
        since = nuevo
    if not filas: return None
    df = pd.DataFrame(filas, columns=['open_time', 'open', 'high', 'low', 'close', 'volume']).drop_duplicates('open_time')
    tam = float(KU.markets[sym].get('contractSize') or 1)
    df['quote_volume'] = df['volume'] * tam * df['close']      # volumen en contratos → USD aproximado
    df['datetime'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
    return df[['datetime', 'open', 'high', 'low', 'close', 'quote_volume']]

def kucoin_funding(sym):
    since = int(T0.timestamp() * 1000); filas = []; fallas = 0
    while since < KU.milliseconds():
        try:
            lote = KU.fetch_funding_rate_history(sym, since=since, limit=100,
                                                 params={'from': since, 'to': min(since + 30 * 86400_000, KU.milliseconds())})
        except Exception:
            fallas += 1; time.sleep(2)
            if fallas > 5: break
            continue
        if lote:
            filas += [(x['timestamp'], x['fundingRate']) for x in lote]
        since += 30 * 86400_000
    if not filas: return None
    df = pd.DataFrame(filas, columns=['ts', 'tasa']).drop_duplicates('ts')
    df['datetime'] = pd.to_datetime(df['ts'], unit='ms', utc=True).dt.round('min')
    return df[['datetime', 'tasa']]

contratos = []
for base, s in KU_USDT.items():
    m = KU.markets[s]; i = m.get('info', {})
    contratos.append(dict(base=base, id=m['id'], tam=m.get('contractSize'), minimo=(m.get('limits', {}).get('amount') or {}).get('min'),
                          tick=i.get('tickSize'), apertura=pd.to_datetime(i.get('firstOpenDate'), unit='ms', utc=True) if i.get('firstOpenDate') else None,
                          comision_taker=i.get('takerFeeRate'), activo=m.get('active')))
pd.DataFrame(contratos).sort_values('base').to_csv(f'{OUT}/contratos_kucoin.csv', index=False)
print('contratos_kucoin.csv:', len(contratos))

## 5) Descargar todo (reanudable)

In [ ]:
PREFIJOS = [('', 1.0), ('1000', 1000.0), ('1000000', 1e6)]

def bajar(row):
    velas, fund, info = [], [], []
    for s, d0, d1 in candidatos(row['slug'], row['symbol']):
        hecho = False
        for pre, fac in PREFIJOS:
            par = f'{pre}{s}USDT'
            if par not in PARES_UM: continue
            v = binance_velas(par)
            if v is None or not len(v := recortar(v, d0, d1)): continue
            for c in ('open', 'high', 'low', 'close'): v[c] = v[c] / fac
            f = binance_funding(par)
            velas.append(v); info.append(dict(fuente='binance', par=par, factor=fac))
            if f is not None and len(f := recortar(f, d0, d1)): fund.append(f)
            hecho = True; break
        if not hecho and s in KU_USDT:
            v = kucoin_velas(KU_USDT[s])
            if v is not None and len(v := recortar(v, d0, d1)):
                velas.append(v); info.append(dict(fuente='kucoin', par=KU_USDT[s], factor=1.0))
                f = kucoin_funding(KU_USDT[s])
                if f is not None and len(f := recortar(f, d0, d1)): fund.append(f)
    if not velas: return None, None, None
    v = pd.concat(velas).drop_duplicates('datetime', keep='last').sort_values('datetime')
    v = v[(v.datetime >= T0) & (v.datetime < T1)].reset_index(drop=True)
    f = pd.concat(fund).drop_duplicates('datetime', keep='last').sort_values('datetime') if fund else None
    if f is not None: f = f[(f.datetime >= T0) & (f.datetime < T1)].reset_index(drop=True)
    return v, f, info

log_path = f'{OUT}/log_perpetuos.csv'
log = pd.read_csv(log_path).to_dict('records') if (os.path.exists(log_path) and not REDESCARGAR) else []
hechas = {r['slug'] + '|' + r['symbol'] for r in log}

for i, row in res.iterrows():
    clave = row['slug'] + '|' + row['symbol']
    if clave in hechas: continue
    t0 = time.time()
    try:
        v, f, info = bajar(row)
    except Exception as err:
        print(f"{i+1:>3}/{len(res)} {row['symbol']:<8} error: {str(err)[:100]}"); continue
    if v is None or v.empty:
        print(f"{i+1:>3}/{len(res)} {row['symbol']:<8} — sin perpetuo")
        log.append(dict(slug=row['slug'], symbol=row['symbol'], velas=0))
    else:
        base = f"{POR_MONEDA}/{row['slug']}__{row['symbol']}"
        v.to_parquet(base + '_velas.parquet', index=False)
        if f is not None and len(f): f.to_parquet(base + '_funding.parquet', index=False)
        log.append(dict(slug=row['slug'], symbol=row['symbol'], velas=len(v), funding=0 if f is None else len(f),
                        fuentes=';'.join(f"{x['fuente']}:{x['par']}" for x in info), desde=str(v.datetime.min()), hasta=str(v.datetime.max())))
        print(f"{i+1:>3}/{len(res)} {row['symbol']:<8} {log[-1]['fuentes']:<32} {len(v):>6} velas {log[-1]['funding']:>5} funding "
              f"{v.datetime.min():%Y-%m-%d} → {v.datetime.max():%Y-%m-%d} ({time.time()-t0:.0f}s)")
    pd.DataFrame(log).to_csv(log_path, index=False)

log = pd.DataFrame(log)
print('\nCon perpetuo:', (log.velas > 0).sum(), '| sin perpetuo:', (log.velas == 0).sum())

## 6) Matrices, listados y chequeos

In [ ]:
ok = log[log.velas > 0].sort_values('velas')
V = pd.concat([pd.read_parquet(f"{POR_MONEDA}/{r.slug}__{r.symbol}_velas.parquet").assign(symbol=r.symbol) for r in ok.itertuples()], ignore_index=True)
salidas = {'open': '_aperturas', 'high': '_maximos', 'low': '_minimos', 'close': '_cierres', 'quote_volume': '_volumen_usdt'}
rutas = []
for col, nombre in salidas.items():
    m = V.pivot_table(index='datetime', columns='symbol', values=col, aggfunc='last').sort_index()
    ruta = f'/content/{nombre}_4h_perp.csv'; m.to_csv(ruta); rutas.append(ruta)
    print(f'{nombre:14s}', m.shape, m.index.min(), '→', m.index.max())
F = []
for r in ok.itertuples():
    p = f"{POR_MONEDA}/{r.slug}__{r.symbol}_funding.parquet"
    if os.path.exists(p): F.append(pd.read_parquet(p).assign(symbol=r.symbol))
F = pd.concat(F, ignore_index=True)
Fm = F.pivot_table(index='datetime', columns='symbol', values='tasa', aggfunc='last').sort_index()
Fm.to_csv('/content/_funding_perp.csv'); rutas.append('/content/_funding_perp.csv')
print('funding', Fm.shape)
lst = V.groupby('symbol').datetime.agg(['min', 'max', 'count']).rename(columns={'min': 'primera_vela', 'max': 'ultima_vela', 'count': 'velas'})
lst = lst.join(ok.set_index('symbol')[['slug', 'fuentes']])
lst.to_csv('/content/listados_perp.csv'); rutas.append('/content/listados_perp.csv')
H = pd.read_csv('/content/_maximos_4h_perp.csv', index_col=0); L = pd.read_csv('/content/_minimos_4h_perp.csv', index_col=0)
C = pd.read_csv('/content/_cierres_4h_perp.csv', index_col=0); O = pd.read_csv('/content/_aperturas_4h_perp.csv', index_col=0)
print('velas con máximo/mínimo inconsistente:', int(((H < C) | (H < O) | (L > C) | (L > O)).sum().sum()))
print('funding fuera de ±1 %:', int((Fm.abs() > 0.01).sum().sum()))

## 7) Huellas SHA-256 y zip para subir

In [ ]:
import shutil
shutil.copy(f'{OUT}/contratos_kucoin.csv', '/content/contratos_kucoin.csv'); rutas.append('/content/contratos_kucoin.csv')
shutil.copy(log_path, '/content/log_perpetuos.csv'); rutas.append('/content/log_perpetuos.csv')
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
manif = dict(creado=str(pd.Timestamp.now('UTC')), desde=DESDE, hasta=HASTA, archivos={os.path.basename(r): sha(r) for r in rutas})
json.dump(manif, open('/content/MANIFIESTO.json', 'w'), indent=1); rutas.append('/content/MANIFIESTO.json')
with zipfile.ZipFile('/content/perpetuos_4h.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for r in rutas: z.write(r, os.path.basename(r))
print('zip:', round(os.path.getsize('/content/perpetuos_4h.zip') / 1e6, 1), 'MB')
from google.colab import files
files.download('/content/perpetuos_4h.zip')